# W06-迭代器与生成器

- Iterable: 可以调用 next(iter)
- Iterator: 自动迭代 iter(it) -> it 不断给你next(it)

In [3]:
for x in "abc":
    print(x)

print("="*10)

it = iter("abc")
while True:
    try:
        x = next(it)
        print(x)
    except StopIteration:
        break
        

a
b
c
a
b
c


## 自己造一个迭代器

In [6]:
# __iter__
# __next__

class CountdownFour:
    def __init__(self, n: int):
        self.n = n

    def __iter__(self):
        return self

    def __next__(self):
        if self.n <= 0:
            raise StopIteration
        cur = self.n
        self.n -= 4
        return cur

for x in CountdownFour(39):
    print(x)
    

39
35
31
27
23
19
15
11
7
3


## 生成器函数

- 普通函数：一次性把结果存起来，占用内存
- 生成器函数（一种迭代器）：用一次输出结果，边产生边给你，更适合数据很多、或你不一定要全用完的情况

In [8]:
def countdown_two(n):
    while n > 0:
        yield n
        n -= 2

for x in countdown_two(7):
    print(x)
    

7
5
3
1


## 生成器表达式

In [10]:
squares_list = [x*x for x in range(10)] # 立刻算 10 个数，占用内存
squares_gen  = (x*x for x in range(10)) # 一个都没算，几乎不占内存
print(sum(squares_gen))

285


In [14]:
## 读一个 1GB 的 CSV，统计第 3 列数字之和，内存不能超过 100MB。

def col3_nums(path):
    with open(path, encoding='utf-8') as f:
        next(f)
        yield float(line.split(",")[2])

# print(sum(col3_nums("big.csv"))

## yield from：委托给"子生成器" 

In [15]:
def chain(*iterables):
    for it in iterables:
        for x in it:
            yield x

def chain2(*iterables):
    for it in iterables:
        yield from it

print(list(chain([1, 2], (3, 4), "calo")))
print(list(chain2([1, 2], (3, 4), "calo")))

[1, 2, 3, 4, 'c', 'a', 'l', 'o']
[1, 2, 3, 4, 'c', 'a', 'l', 'o']


## 协程基础：`send()` / `throw()` / `close()`

In [17]:
def echo():
    while True:
        msg = yield
        print(f"收到：{msg}")


g = echo()
next(g) # 必须先"启动"到第一个 yield
g.send("hello")
g.send("who are you?")
g.close() # 关闭生成器

收到：hello
收到：who are you?


In [18]:
from itertools import product
code = ["".join(p) for p in product("0123456789", repeat=3)]
print(len(code))
print(code[:5])

1000
['000', '001', '002', '003', '004']


## 实战：手写 itertools 核心函数

In [20]:
def my_range(start, stop=None, step=2):
    if stop is None:
        start, stop = 0, start
    cur = start
    while (step > 0 and cur < stop) or (step < 0 and cur > stop):
        yield cur
        cur += step

def my_enumerate(iterable, start=0):
    i = start
    for x in iterable:
        yield i, x
        i += 1

def my_zip(*iterables):
    iters = [iter(it) for it in iterables]
    while True:
        result = []
        for it in iters:
            try:
                result.append(next(it))
            except StopIteration:
                return
        yield tuple(result)

def my_chain(*iterables):
    for it in iterables:
        yield from it

print(list(my_range(5)))                              # [0, 2, 4]
print(list(my_enumerate("abc", 1)))                   # [(1,'a'),(2,'b'),(3,'c')]
print(list(my_zip([1,2,3], "abc", [10,20,30])))       # [(1,'a',10),(2,'b',20),(3,'c',30)]
print(list(my_chain([1], (2,3), "ab")))               # [1, 2, 3, 'a', 'b']


[0, 2, 4]
[(1, 'a'), (2, 'b'), (3, 'c')]
[(1, 'a', 10), (2, 'b', 20), (3, 'c', 30)]
[1, 2, 3, 'a', 'b']


## 实战：1GB 大文件按行去重

In [21]:
def unique_lines(path):
    seen = set()
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if line not in seen:
                seen.add(line)
                yield line